# nb_03 — DimSupplier

**Purpose:** Load `DimSupplier` from `Files/seed_data/dim_supplier.csv` with SCD Type 2 scaffold columns.

## Overview
- Reads `dim_supplier.csv`
- Adds SCD2 tracking columns: `scd_start_date`, `scd_end_date`, `is_current`
- Writes to Delta table `DimSupplier` (overwrite for initial load)
- Includes `upsert_scd2` helper for incremental updates

**Prerequisite:** nb_00_setup_lakehouse must be run first.

In [ ]:
from pyspark.sql.functions import current_date, lit

# Read seed CSV
df = spark.read.csv(
    "Files/seed_data/dim_supplier.csv",
    header=True,
    inferSchema=True
)

# Add SCD2 scaffold columns
df = (
    df
    .withColumn("scd_start_date", current_date())
    .withColumn("scd_end_date", lit("9999-12-31").cast("date"))
    .withColumn("is_current", lit(True))
)

print(f"Rows read: {df.count()}")
df.printSchema()

In [ ]:
# Write to Delta table (initial full load)
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("DimSupplier")
)

print("DimSupplier loaded successfully.")
spark.sql("SELECT COUNT(*) AS row_count FROM DimSupplier").show()

In [ ]:
# SCD2 upsert helper — use for incremental loads
from delta.tables import DeltaTable

def upsert_scd2(spark, table, source_df, nk, tracked_cols):
    """Merge source_df into an existing SCD2 Delta table.

    Parameters
    ----------
    table        : str  — Delta table name
    source_df    : DataFrame — incoming changed records
    nk           : str  — natural key column name
    tracked_cols : list — columns to watch for changes
    """
    dt = DeltaTable.forName(spark, table)
    (
        dt.alias("t")
        .merge(
            source_df.alias("s"),
            f"t.{nk} = s.{nk} AND t.is_current = true"
        )
        .whenMatchedUpdate(
            condition=" OR ".join(f"s.{c} != t.{c}" for c in tracked_cols),
            set={"scd_end_date": "current_date()", "is_current": "false"}
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

print("upsert_scd2 helper defined. Use for incremental supplier loads.")